# Module 2 - Detecting Epistemic Enclaves

**Time:** 11:15-12:45  
**Tool focus:** CDlib

This notebook compares several community-detection algorithms and then evaluates the resulting clusters as potential epistemic enclaves.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / ".cache"
(CACHE_DIR / "matplotlib").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

import networkx as nx
import numpy as np
import pandas as pd
from cdlib import algorithms
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score


In [ ]:
def build_demo_graph(seed=42):
    rng = np.random.default_rng(seed)
    communities = [
        {"label": "left-mainstream", "camp": "left", "size": 36, "mean_opinion": -0.55, "enclave": 0},
        {"label": "left-enclave", "camp": "left", "size": 18, "mean_opinion": -0.88, "enclave": 1},
        {"label": "right-mainstream", "camp": "right", "size": 36, "mean_opinion": 0.55, "enclave": 0},
        {"label": "right-enclave", "camp": "right", "size": 18, "mean_opinion": 0.88, "enclave": 1},
    ]
    sizes = [community["size"] for community in communities]
    probability_matrix = [
        [0.18, 0.08, 0.02, 0.004],
        [0.08, 0.25, 0.006, 0.001],
        [0.02, 0.006, 0.18, 0.08],
        [0.004, 0.001, 0.08, 0.25],
    ]

    graph = nx.stochastic_block_model(sizes, probability_matrix, seed=seed)
    graph.graph.clear()

    node_id = 0
    for community in communities:
        for _ in range(community["size"]):
            graph.nodes[node_id]["label"] = f"user_{node_id:03d}"
            graph.nodes[node_id]["community_label"] = community["label"]
            graph.nodes[node_id]["camp"] = community["camp"]
            graph.nodes[node_id]["opinion"] = float(np.clip(rng.normal(community["mean_opinion"], 0.09), -1.0, 1.0))
            graph.nodes[node_id]["enclave"] = int(community["enclave"])
            graph.nodes[node_id]["activity"] = int(rng.poisson(9 if community["enclave"] else 6) + 1)
            node_id += 1

    return graph


def write_demo_graph_files(seed=42):
    DATA_RAW.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    graph = build_demo_graph(seed=seed)

    node_frame = pd.DataFrame(
        [{"node_id": node, **attrs} for node, attrs in graph.nodes(data=True)]
    )
    edge_frame = pd.DataFrame(
        [{"source": source, "target": target} for source, target in graph.edges()]
    )

    nx.write_graphml(graph, DATA_RAW / "workshop_network.graphml")
    nx.write_gexf(graph, DATA_RAW / "workshop_network.gexf")
    nx.write_edgelist(graph, DATA_RAW / "workshop_network.edgelist", data=False)
    node_frame.to_csv(DATA_RAW / "workshop_nodes.csv", index=False)
    edge_frame.to_csv(DATA_PROCESSED / "workshop_edges.csv", index=False)


def load_graph(path):
    path = Path(path)
    if path.suffix == ".graphml":
        graph = nx.read_graphml(path)
    elif path.suffix == ".gexf":
        graph = nx.read_gexf(path)
    else:
        graph = nx.read_edgelist(path, nodetype=int)
        node_frame = pd.read_csv(DATA_RAW / "workshop_nodes.csv")
        attributes = node_frame.set_index("node_id").to_dict(orient="index")
        nx.set_node_attributes(graph, attributes)

    graph = nx.convert_node_labels_to_integers(graph, label_attribute="original_id")
    for _, attrs in graph.nodes(data=True):
        attrs["opinion"] = float(attrs["opinion"])
        attrs["activity"] = int(attrs["activity"])
        attrs["enclave"] = int(attrs["enclave"])
    return graph


In [ ]:
write_demo_graph_files()
G = load_graph(DATA_RAW / "workshop_network.graphml")


## 1. Run several community-detection algorithms


In [ ]:
louvain = algorithms.louvain(G)
leiden = algorithms.leiden(G)
infomap = algorithms.infomap(G)
label_prop = algorithms.label_propagation(G)

pd.DataFrame(
    [
        {"algorithm": "Louvain", "communities": len(louvain.communities)},
        {"algorithm": "Leiden", "communities": len(leiden.communities)},
        {"algorithm": "Infomap", "communities": len(infomap.communities)},
        {"algorithm": "Label propagation", "communities": len(label_prop.communities)},
    ]
)


In [ ]:
def partition_frame(partition, column_name):
    rows = []
    for cluster_id, community in enumerate(partition.communities):
        for node in community:
            rows.append({"node_id": int(node), column_name: cluster_id})
    frame = pd.DataFrame(rows)
    node_attributes = pd.DataFrame(
        [
            {
                "node_id": node,
                "camp": G.nodes[node]["camp"],
                "community_label": G.nodes[node]["community_label"],
                "enclave": G.nodes[node]["enclave"],
            }
            for node in G.nodes()
        ]
    )
    return node_attributes.merge(frame, on="node_id", how="left")


In [ ]:
louvain_frame = partition_frame(louvain, "cluster")
leiden_frame = partition_frame(leiden, "cluster")
infomap_frame = partition_frame(infomap, "cluster")
label_prop_frame = partition_frame(label_prop, "cluster")

pd.DataFrame(
    [
        {
            "algorithm": "Louvain vs Louvain",
            "ARI": 1.0,
            "NMI": 1.0,
        },
        {
            "algorithm": "Leiden vs Louvain",
            "ARI": round(adjusted_rand_score(louvain_frame["cluster"], leiden_frame["cluster"]), 4),
            "NMI": round(normalized_mutual_info_score(louvain_frame["cluster"], leiden_frame["cluster"]), 4),
        },
        {
            "algorithm": "Infomap vs Louvain",
            "ARI": round(adjusted_rand_score(louvain_frame["cluster"], infomap_frame["cluster"]), 4),
            "NMI": round(normalized_mutual_info_score(louvain_frame["cluster"], infomap_frame["cluster"]), 4),
        },
        {
            "algorithm": "Label propagation vs Louvain",
            "ARI": round(adjusted_rand_score(louvain_frame["cluster"], label_prop_frame["cluster"]), 4),
            "NMI": round(normalized_mutual_info_score(louvain_frame["cluster"], label_prop_frame["cluster"]), 4),
        },
    ]
)


## 2. Global scale: assortativity


In [ ]:
camp_assortativity = nx.attribute_assortativity_coefficient(G, "camp")
enclave_assortativity = nx.attribute_assortativity_coefficient(
    nx.relabel_nodes(G, {node: node for node in G.nodes()}), "enclave"
)

pd.Series(
    {
        "camp_assortativity": round(camp_assortativity, 4),
        "enclave_assortativity": round(enclave_assortativity, 4),
    }
)


## 3. Meso scale: purity and separation


In [ ]:
community_rows = []
for cluster_id, chunk in louvain_frame.groupby("cluster"):
    nodes = set(chunk["node_id"])
    subgraph = G.subgraph(nodes)
    internal_edges = subgraph.number_of_edges()
    external_edges = nx.cut_size(G, nodes)
    dominant_camp = chunk["camp"].mode().iat[0]
    purity = (chunk["camp"] == dominant_camp).mean()
    separation = internal_edges / max(internal_edges + external_edges, 1)
    community_rows.append(
        {
            "cluster": cluster_id,
            "size": len(nodes),
            "purity": round(float(purity), 4),
            "internal_edges": internal_edges,
            "external_edges": int(external_edges),
            "separation": round(float(separation), 4),
        }
    )

pd.DataFrame(community_rows).sort_values("cluster").reset_index(drop=True)


## 4. Local scale: node conformity


In [ ]:
conformity_rows = []
for node in G.nodes():
    neighbors = list(G.neighbors(node))
    same_neighbor_share = np.nan
    if neighbors:
        same_neighbor_share = np.mean(
            [G.nodes[neighbor]["camp"] == G.nodes[node]["camp"] for neighbor in neighbors]
        )
    conformity_rows.append(
        {
            "node_id": node,
            "camp": G.nodes[node]["camp"],
            "community_label": G.nodes[node]["community_label"],
            "same_neighbor_share": same_neighbor_share,
        }
    )

conformity = pd.DataFrame(conformity_rows)
conformity.groupby("camp")["same_neighbor_share"].describe().round(3)
